In [1]:
import os
import sys

# 【必须放在最开头，import pyspark 之前】
os.environ["JAVA_HOME"] = r"D:\JDK8"
os.environ["HADOOP_HOME"] = r"E:\hadoop\hadoop-3.3.5"
os.environ["JAVA_TOOL_OPTIONS"] = "-Djava.net.preferIPv4Stack=true"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

java_bin = os.path.join(os.environ["JAVA_HOME"], "bin")
hadoop_bin = os.path.join(os.environ["HADOOP_HOME"], "bin")
os.environ["PATH"] = f"{java_bin};{hadoop_bin};" + os.environ.get("PATH", "")

from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.sql.functions import col, when

# 创建SparkSession
spark = (SparkSession.builder
         .appName("MLFeatureEngineering")
         .master("local[1]")
         .config("spark.driver.host", "127.0.0.1")
         .config("spark.driver.bindAddress", "127.0.0.1")
         .config("spark.python.worker.reuse", "false")
         .config("spark.network.timeout", "600s")
         .getOrCreate())

print("java.library.path =", spark._jvm.java.lang.System.getProperty("java.library.path"))

# ====================== 1. 读取三张表CSV数据 ======================
# 读取电影表（只保留有效评分数据）
df_movie_all = spark.read.option("header", "true").csv(r"E:/movie_analysiz_01/test02/output/data/movie/clean/movies")
df_movie = df_movie_all.filter(col("SCORE_FLAG") == "valid")

# 读取评分表rating
df_rating = spark.read.option("header", "true").csv(r"E:/movie_analysiz_01/test02/output/data/movie/clean/ratings")

# 读取评论表comment
df_comment = spark.read.option("header", "true").csv(r"E:/movie_analysiz_01/test02/output/data/movie/clean/comments")

print("=" * 70)
print(f"【电影表】总数据：{df_movie_all.count()} 条，有效评分数据：{df_movie.count()} 条")
print(f"【评分表】总记录：{df_rating.count()} 条")
print(f"【评论表】总记录：{df_comment.count()} 条")
print("=" * 70)

# ====================== 空值填充：修正真实字段名 GENRES_CLEAN / REGIONS_CLEAN / LANGUAGES_CLEAN / TAGS ======================
# movie表类别字段填充（使用清洗后的标准化字段，匹配数据表真实列）
df_movie = df_movie.withColumn("GENRES_CLEAN",
                               when(col("GENRES_CLEAN") == "", "unknown").otherwise(col("GENRES_CLEAN")))
df_movie = df_movie.withColumn("REGIONS_CLEAN",
                               when(col("REGIONS_CLEAN") == "", "unknown").otherwise(col("REGIONS_CLEAN")))
df_movie = df_movie.withColumn("LANGUAGES_CLEAN",
                               when(col("LANGUAGES_CLEAN") == "", "unknown").otherwise(col("LANGUAGES_CLEAN")))
df_movie = df_movie.withColumn("TAGS", when(col("TAGS") == "", "unknown").otherwise(col("TAGS")))

# rating表用户ID填充空值
df_rating = df_rating.withColumn("USER_MD5", when(col("USER_MD5").isNull(), "unknown").otherwise(col("USER_MD5")))
df_rating = df_rating.withColumn("MOVIE_ID", when(col("MOVIE_ID").isNull(), "unknown").otherwise(col("MOVIE_ID")))

# comment表ID字段填充空值
df_comment = df_comment.withColumn("USER_MD5", when(col("USER_MD5").isNull(), "unknown").otherwise(col("USER_MD5")))
df_comment = df_comment.withColumn("MOVIE_ID", when(col("MOVIE_ID").isNull(), "unknown").otherwise(col("MOVIE_ID")))


# ====================== 3.5.2 类别特征编码通用封装函数 ======================
def encode_category_feature(df, input_col, idx_col, vec_col):
    """
    类别特征编码：StringIndexer + OneHotEncoder
    :param df: 待处理DataFrame
    :param input_col: 原始类别字段名
    :param idx_col: 索引输出字段名
    :param vec_col: 独热向量输出字段名
    :return: 编码完成后的DataFrame
    """
    # 1. 字符串转数字索引
    indexer = StringIndexer(inputCol=input_col, outputCol=idx_col)
    df_idx = indexer.fit(df).transform(df)
    # 2. 索引转独热编码
    encoder = OneHotEncoder(inputCol=idx_col, outputCol=vec_col, dropLast=True)
    df_encoded = encoder.fit(df_idx).transform(df_idx)
    return df_encoded


# ====================== 一、movie电影表 多类别字段编码（使用正确标准化字段） ======================
print("\n" + "=" * 70)
print("【1. 电影表movie 类别特征编码】")
print("=" * 70)

# 1. 电影类型 GENRES_CLEAN
df_movie = encode_category_feature(df_movie, "GENRES_CLEAN", "genres_idx", "genres_onehot")
# 2. 制片地区 REGIONS_CLEAN
df_movie = encode_category_feature(df_movie, "REGIONS_CLEAN", "region_idx", "region_onehot")
# 3. 影片语言 LANGUAGES_CLEAN
df_movie = encode_category_feature(df_movie, "LANGUAGES_CLEAN", "language_idx", "language_onehot")
# 4. 标签 TAGS
df_movie = encode_category_feature(df_movie, "TAGS", "tags_idx", "tags_onehot")

# 展示电影表编码结果
df_movie.select("movie_id", "GENRES_CLEAN", "genres_onehot", "REGIONS_CLEAN", "region_onehot", "douban_score").show(
    truncate=False)

# ====================== 二、rating评分表 类别特征编码 ======================
print("\n" + "=" * 70)
print("【2. 评分表rating 类别特征编码】")
print("=" * 70)

# 用户ID USER_MD5 编码
df_rating = encode_category_feature(df_rating, "USER_MD5", "user_idx", "user_onehot")
# 电影ID MOVIE_ID 编码
df_rating = encode_category_feature(df_rating, "MOVIE_ID", "movie_rating_idx", "movie_rating_onehot")

df_rating.select("RATING_ID", "USER_MD5", "user_onehot", "MOVIE_ID", "movie_rating_onehot", "RATING").show(
    truncate=False)

# ====================== 三、comment评论表 类别特征编码 ======================
print("\n" + "=" * 70)
print("【3. 评论表comment 类别特征编码】")
print("=" * 70)

# 用户ID USER_MD5 编码
df_comment = encode_category_feature(df_comment, "USER_MD5", "comment_user_idx", "comment_user_onehot")
# 电影ID MOVIE_ID 编码
df_comment = encode_category_feature(df_comment, "MOVIE_ID", "comment_movie_idx", "comment_movie_onehot")

df_comment.select("COMMENT_ID", "USER_MD5", "comment_user_onehot", "MOVIE_ID", "comment_movie_onehot", "VOTES").show(
    truncate=False)

print("\n✅ 三张表全部类别特征编码完成！")
# spark.stop()

java.library.path = D:\JDK8\bin;C:\Windows\Sun\Java\bin;C:\Windows\system32;C:\Windows;D:\JDK8\bin;E:\hadoop\hadoop-3.3.5\bin;E:\HDFS Python\HDFSpythonProject\.venv\Scripts;D:\JDK8\bin;D:\Engineering Software\VMware\bin\;C:\Program Files\Common Files\Oracle\Java\javapath;C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.1\bin;C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.1\libnvvp;D:\编程软件\Python\Scripts\;E:\hadoop\hadoop-3.3.5\bin;E:\HDFSPython\HDFSpythonProject\.venv\Lib\site-packages\pyspark\bin;程软件;Python\;Program Files\Microsoft\jdk-11.0.16.101-hotspot\bin;C:\Windows\system32;C:\Windows;C:\Windows\System32\Wbem;C:\Windows\System32\WindowsPowerShell\v1.0\;C:\Windows\System32\OpenSSH\;C:\Program Files (x86)\NVIDIA Corporation\PhysX\Common;C:\Program Files\dotnet\;C:\Program Files (x86)\Windows Kits\10\Windows Performance Toolkit\;rogram Files\NVIDIA Corporation\NVIDIA app\NvDLISR;D:\AppDownload\Git\cmd;D:\Program Files\pcsuite\;C:\Program Files\NVIDIA Corporation\NVIDIA

In [1]:
# spark.stop()

java.library.path = D:\JDK8\bin;C:\Windows\Sun\Java\bin;C:\Windows\system32;C:\Windows;D:\JDK8\bin;E:\hadoop\hadoop-3.3.5\bin;E:\HDFS Python\HDFSpythonProject\.venv\Scripts;D:\JDK8\bin;D:\Engineering Software\VMware\bin\;C:\Program Files\Common Files\Oracle\Java\javapath;C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.1\bin;C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.1\libnvvp;D:\编程软件\Python\Scripts\;E:\hadoop\hadoop-3.3.5\bin;E:\HDFSPython\HDFSpythonProject\.venv\Lib\site-packages\pyspark\bin;程软件;Python\;Program Files\Microsoft\jdk-11.0.16.101-hotspot\bin;C:\Windows\system32;C:\Windows;C:\Windows\System32\Wbem;C:\Windows\System32\WindowsPowerShell\v1.0\;C:\Windows\System32\OpenSSH\;C:\Program Files (x86)\NVIDIA Corporation\PhysX\Common;C:\Program Files\dotnet\;C:\Program Files (x86)\Windows Kits\10\Windows Performance Toolkit\;rogram Files\NVIDIA Corporation\NVIDIA app\NvDLISR;D:\AppDownload\Git\cmd;D:\Program Files\pcsuite\;C:\Program Files\NVIDIA Corporation\NVIDIA

In [ ]:
import os
import sys

# ========== 环境变量配置 ==========
os.environ["JAVA_HOME"] = r"D:\JDK8"
os.environ["HADOOP_HOME"] = r"E:\hadoop\hadoop-3.3.5"
os.environ["JAVA_TOOL_OPTIONS"] = "-Djava.net.preferIPv4Stack=true"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

java_bin = os.path.join(os.environ["JAVA_HOME"], "bin")
hadoop_bin = os.path.join(os.environ["HADOOP_HOME"], "bin")
os.environ["PATH"] = f"{java_bin};{hadoop_bin};" + os.environ.get("PATH", "")

# ========== 导入依赖包 ==========
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.sql.functions import col, coalesce, lit
from pyspark.sql.types import DoubleType

# ========== 创建Spark会话（内存扩容+GC优化） ==========
spark = (SparkSession.builder
         .appName("ML_Feature_Pipeline_Full")
         .master("local[*]")
         .config("spark.driver.memory", "6g")
         .config("spark.executor.memory", "6g")
         .config("spark.executor.extraJavaOptions", "-XX:+UseG1GC -XX:MaxGCPauseMillis=200")
         .config("spark.driver.extraJavaOptions", "-XX:+UseG1GC -XX:MaxGCPauseMillis=200")
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
         .config("spark.sql.adaptive.coalescePartitions.minPartitionNum", "4")
         .config("spark.sql.files.maxPartitionBytes", "128m")
         .config("spark.driver.host", "127.0.0.1")
         .config("spark.driver.bindAddress", "127.0.0.1")
         .config("spark.python.worker.reuse", "false")
         .config("spark.network.timeout", "600s")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("java内存配置加载完成：", spark._jvm.java.lang.System.getProperty("java.library.path"))

# ========== 读取三张原始数据表 ==========
df_movie_all = spark.read.option("header", "true").csv(r"E:/movie_analysiz_01/test02/output/data/movie/clean/movies")
df_movie_raw = df_movie_all.filter(col("SCORE_FLAG") == "valid")

df_rating_raw = spark.read.option("header", "true").csv(r"E:/movie_analysiz_01/test02/output/data/movie/clean/ratings")

df_comment_raw = spark.read.option("header", "true").csv(
    r"E:/movie_analysiz_01/test02/output/data/movie/clean/comments")

print("=" * 70)
print(f"【电影表】总数据：{df_movie_all.count()} 条，有效建模数据：{df_movie_raw.count()} 条")
print(f"【评分表】总记录：{df_rating_raw.count()} 条")
print(f"【评论表】总记录：{df_comment_raw.count()} 条")
print("=" * 70)

# ========== 空值填充 + 字符串转数值 ==========
# movie表
df_movie_raw = df_movie_raw.withColumn("GENRES_CLEAN", coalesce(col("GENRES_CLEAN"), lit("unknown")))
df_movie_raw = df_movie_raw.withColumn("REGIONS_CLEAN", coalesce(col("REGIONS_CLEAN"), lit("unknown")))
df_movie_raw = df_movie_raw.withColumn("LANGUAGES_CLEAN", coalesce(col("LANGUAGES_CLEAN"), lit("unknown")))
df_movie_raw = df_movie_raw.withColumn("TAGS", coalesce(col("TAGS"), lit("unknown")))
df_movie_raw = df_movie_raw.withColumn("YEAR", coalesce(col("YEAR"), lit("0")).cast(DoubleType()))
df_movie_raw = df_movie_raw.withColumn("DOUBAN_VOTES", coalesce(col("DOUBAN_VOTES"), lit("0")).cast(DoubleType()))
df_movie_raw = df_movie_raw.withColumn("MINS", coalesce(col("MINS"), lit("0")).cast(DoubleType()))

# rating表
df_rating_raw = df_rating_raw.withColumn("USER_MD5", coalesce(col("USER_MD5"), lit("unknown")))
df_rating_raw = df_rating_raw.withColumn("MOVIE_ID", coalesce(col("MOVIE_ID"), lit("unknown")))
df_rating_raw = df_rating_raw.withColumn("RATING", coalesce(col("RATING"), lit("0")).cast(DoubleType()))

# comment表
df_comment_raw = df_comment_raw.withColumn("USER_MD5", coalesce(col("USER_MD5"), lit("unknown")))
df_comment_raw = df_comment_raw.withColumn("MOVIE_ID", coalesce(col("MOVIE_ID"), lit("unknown")))
df_comment_raw = df_comment_raw.withColumn("VOTES", coalesce(col("VOTES"), lit("0")).cast(DoubleType()))
df_comment_raw = df_comment_raw.withColumn("RATING", coalesce(col("RATING"), lit("0")).cast(DoubleType()))

# ====================== 一、电影表Pipeline构建与训练 ======================
print("\n" + "=" * 70)
print("【电影表 构建Pipeline流水线】")
print("=" * 70)
genres_index = StringIndexer(inputCol="GENRES_CLEAN", outputCol="genres_idx")
genres_onehot = OneHotEncoder(inputCol="genres_idx", outputCol="genres_vec", dropLast=True)

region_index = StringIndexer(inputCol="REGIONS_CLEAN", outputCol="region_idx")
region_onehot = OneHotEncoder(inputCol="region_idx", outputCol="region_vec", dropLast=True)

lang_index = StringIndexer(inputCol="LANGUAGES_CLEAN", outputCol="lang_idx")
lang_onehot = OneHotEncoder(inputCol="lang_idx", outputCol="lang_vec", dropLast=True)

tags_index = StringIndexer(inputCol="TAGS", outputCol="tags_idx")
tags_onehot = OneHotEncoder(inputCol="tags_idx", outputCol="tags_vec", dropLast=True)

num_assembler = VectorAssembler(inputCols=["YEAR", "DOUBAN_VOTES", "MINS"], outputCol="num_raw_vec",
                                handleInvalid="keep")
scaler = StandardScaler(inputCol="num_raw_vec", outputCol="num_scaled_vec", withMean=True, withStd=True)
total_assembler = VectorAssembler(inputCols=["genres_vec", "region_vec", "lang_vec", "tags_vec", "num_scaled_vec"],
                                  outputCol="features", handleInvalid="keep")

movie_stages = [genres_index, genres_onehot, region_index, region_onehot, lang_index, lang_onehot, tags_index,
                tags_onehot, num_assembler, scaler, total_assembler]
movie_pipeline = Pipeline(stages=movie_stages)
movie_pipe_model = movie_pipeline.fit(df_movie_raw)
df_movie_final = movie_pipe_model.transform(df_movie_raw)

print("电影表Pipeline处理完成，预览前20行数据：")
df_movie_final.select("movie_id", "GENRES_CLEAN", "features", "DOUBAN_SCORE").show(20)

# ====================== 二、评分表Pipeline构建与训练 ======================
print("\n" + "=" * 70)
print("【评分表 构建Pipeline流水线】")
print("=" * 70)
user_index_r = StringIndexer(inputCol="USER_MD5", outputCol="user_idx_r")
user_onehot_r = OneHotEncoder(inputCol="user_idx_r", outputCol="user_vec_r", dropLast=True)
mid_index_r = StringIndexer(inputCol="MOVIE_ID", outputCol="mid_idx_r")
mid_onehot_r = OneHotEncoder(inputCol="mid_idx_r", outputCol="mid_vec_r", dropLast=True)

num_assembler_r = VectorAssembler(inputCols=["RATING"], outputCol="num_raw_r", handleInvalid="keep")
scaler_r = StandardScaler(inputCol="num_raw_r", outputCol="num_scaled_r", withMean=True, withStd=True)
total_assembler_r = VectorAssembler(inputCols=["user_vec_r", "mid_vec_r", "num_scaled_r"], outputCol="features",
                                    handleInvalid="keep")

rating_stages = [user_index_r, user_onehot_r, mid_index_r, mid_onehot_r, num_assembler_r, scaler_r, total_assembler_r]
rating_pipeline = Pipeline(stages=rating_stages)
rating_pipe_model = rating_pipeline.fit(df_rating_raw)
df_rating_final = rating_pipe_model.transform(df_rating_raw)

print("评分表Pipeline处理完成，预览前20行数据：")
df_rating_final.select("RATING_ID", "USER_MD5", "features").show(20)

# ====================== 三、评论表Pipeline构建与训练 ======================
print("\n" + "=" * 70)
print("【评论表 构建Pipeline流水线】")
print("=" * 70)
user_index_c = StringIndexer(inputCol="USER_MD5", outputCol="user_idx_c")
user_onehot_c = OneHotEncoder(inputCol="user_idx_c", outputCol="user_vec_c", dropLast=True)
mid_index_c = StringIndexer(inputCol="MOVIE_ID", outputCol="mid_idx_c")
mid_onehot_c = OneHotEncoder(inputCol="mid_idx_c", outputCol="mid_vec_c", dropLast=True)

num_assembler_c = VectorAssembler(inputCols=["VOTES", "RATING"], outputCol="num_raw_c", handleInvalid="keep")
scaler_c = StandardScaler(inputCol="num_raw_c", outputCol="num_scaled_c", withMean=True, withStd=True)
total_assembler_c = VectorAssembler(inputCols=["user_vec_c", "mid_vec_c", "num_scaled_c"], outputCol="features",
                                    handleInvalid="keep")

comment_stages = [user_index_c, user_onehot_c, mid_index_c, mid_onehot_c, num_assembler_c, scaler_c, total_assembler_c]
comment_pipeline = Pipeline(stages=comment_stages)
comment_pipe_model = comment_pipeline.fit(df_comment_raw)
df_comment_final = comment_pipe_model.transform(df_comment_raw)

print("评论表Pipeline处理完成，预览前20行数据：")
df_comment_final.select("COMMENT_ID", "USER_MD5", "features").show(20)

print("\n🎉 特征工程全部完成，变量已生成：")
print("df_movie_final 电影特征数据集 | movie_pipeline 电影流水线")
print("df_rating_final 评分特征数据集 | rating_pipeline 评分流水线")
print("df_comment_final 评论特征数据集 | comment_pipeline 评论流水线")

In [4]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.functions import when

print("=" * 60)
print("5. 逻辑回归模型")
print("=" * 60)

# 创建标签：评分 >= 9.0 为高质量电影（1），否则为0
df_labeled = df.withColumn("label",
    when(col("rating") >= 9.0, 1.0).otherwise(0.0))

# 重新构建包含标签的特征工程Pipeline
pipeline_with_label = Pipeline(stages=[
    genre1_indexer,
    genre2_indexer,
    genre1_encoder,
    genre2_encoder,
    assembler,
    scaler
])

pipeline_model = pipeline_with_label.fit(df_labeled)
feature_df = pipeline_model.transform(df_labeled)

# 划分训练集和测试集
train_data, test_data = feature_df.randomSplit([0.7, 0.3], seed=42)
print(f"训练集: {train_data.count()}, 测试集: {test_data.count()}")

# 训练逻辑回归模型
lr = LogisticRegression(featuresCol="features_scaled", labelCol="label",
                        maxIter=100, regParam=0.01)

lr_model = lr.fit(train_data)
lr_predictions = lr_model.transform(test_data)

# 评估模型
# 二分类评估器（AUC）：输出 AUC 值，范围 0~1，越高表示模型区分正负类的能力越强
bin_eval = BinaryClassificationEvaluator(labelCol="label",
                                         rawPredictionCol="rawPrediction",
                                         metricName="areaUnderROC")
auc = bin_eval.evaluate(lr_predictions)

# 多分类评估器（准确率、F1）
multi_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")
accuracy = multi_eval.setMetricName("accuracy").evaluate(lr_predictions)
f1 = multi_eval.setMetricName("f1").evaluate(lr_predictions)

print(f"逻辑回归评估结果:")
print(f"  AUC: {auc:.4f}")
print(f"  准确率: {accuracy:.4f}")
print(f"  F1分数: {f1:.4f}")

# 混淆矩阵
print("\n混淆矩阵 (label, prediction):")
lr_predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

5. 逻辑回归模型
训练集: 5, 测试集: 2
逻辑回归评估结果:
  AUC: 1.0000
  准确率: 1.0000
  F1分数: 1.0000

混淆矩阵 (label, prediction):
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|    1|
|  1.0|       1.0|    1|
+-----+----------+-----+



In [5]:
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier

print("=" * 60)
print("6. 决策树与随机森林")
print("=" * 60)

# 决策树
dt = DecisionTreeClassifier(featuresCol="features_scaled", labelCol="label",
                            maxDepth=5, impurity="gini")

dt_model = dt.fit(train_data)
dt_predictions = dt_model.transform(test_data)

# 评估决策树
dt_auc = bin_eval.evaluate(dt_predictions)
dt_accuracy = multi_eval.setMetricName("accuracy").evaluate(dt_predictions)
dt_f1 = multi_eval.setMetricName("f1").evaluate(dt_predictions)

print("决策树评估结果:")
print(f"  AUC: {dt_auc:.4f}")
print(f"  准确率: {dt_accuracy:.4f}")
print(f"  F1分数: {dt_f1:.4f}")

# 决策树特征重要性 输出为 (11,[1],[1.0])，表示只有索引 1 的特征（对应 rating特征列）重要性为 1.0，其余均为 0
# 索引 0: year
# 索引 1: rating
# 索引 2: votes
# 索引 3~5: genre1_onehot
# 索引 6~10: genre2_onehot
print(f"决策树特征重要性: {dt_model.featureImportances}")

# 随机森林
rf = RandomForestClassifier(featuresCol="features_scaled", labelCol="label",
                            numTrees=100, maxDepth=10, impurity="gini")

rf_model = rf.fit(train_data)
rf_predictions = rf_model.transform(test_data)

# 评估随机森林
rf_auc = bin_eval.evaluate(rf_predictions)
rf_accuracy = multi_eval.setMetricName("accuracy").evaluate(rf_predictions)
rf_f1 = multi_eval.setMetricName("f1").evaluate(rf_predictions)

print("\n随机森林评估结果:")
print(f"  AUC: {rf_auc:.4f}")
print(f"  准确率: {rf_accuracy:.4f}")
print(f"  F1分数: {rf_f1:.4f}")

# 有多个特征：通过引入随机性降低单棵树的方差，让模型不那么依赖单一特征
print(f"随机森林特征重要性: {rf_model.featureImportances}")

6. 决策树与随机森林
决策树评估结果:
  AUC: 1.0000
  准确率: 1.0000
  F1分数: 1.0000
决策树特征重要性: (11,[1],[1.0])

随机森林评估结果:
  AUC: 1.0000
  准确率: 1.0000
  F1分数: 1.0000
随机森林特征重要性: (11,[0,1,2,3,4,5,6,9,10],[0.2462709284627093,0.3717851706892803,0.10817677756033921,0.0022831050228310445,0.06316590563165905,0.1129267232006958,0.02811589475973038,0.04729832572298325,0.019977168949771692])


In [6]:
print("=" * 60)
print("7. 模型对比与保存")
print("=" * 60)

# 汇总对比
models = [
    ("逻辑回归", auc, accuracy, f1),
    ("决策树", dt_auc, dt_accuracy, dt_f1),
    ("随机森林", rf_auc, rf_accuracy, rf_f1),
]

print("模型性能对比:")
print("+" + "-" * 60 + "+")
print("| 模型名称   | AUC    | 准确率  | F1分数  |")
print("+" + "-" * 60 + "+")
for name, auc_score, acc, f1_score in models:
    print(f"| {name:<10} | {auc_score:.4f} | {acc:.4f}   | {f1_score:.4f}   |")
print("+" + "-" * 60 + "+")

# 保存模型
print("\n保存模型到 ./models/")
rf_model.write().overwrite().save("./models/rf_model2")
print("模型保存完成！")

# 加载模型
from pyspark.ml.classification import RandomForestClassificationModel

loaded_model = RandomForestClassificationModel.load("./models/rf_model2")
print("模型加载成功！")

7. 模型对比与保存
模型性能对比:
+------------------------------------------------------------+
| 模型名称   | AUC    | 准确率  | F1分数  |
+------------------------------------------------------------+
| 逻辑回归       | 1.0000 | 1.0000   | 1.0000   |
| 决策树        | 1.0000 | 1.0000   | 1.0000   |
| 随机森林       | 1.0000 | 1.0000   | 1.0000   |
+------------------------------------------------------------+

保存模型到 ./models/
模型保存完成！
模型加载成功！


In [7]:
from pyspark.ml.regression import RandomForestRegressor, LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml import Pipeline

print("=" * 60)
print("8. 电影评分预测（回归任务）")
print("=" * 60)

# 准备数据：使用年份、投票数、类型预测评分
# 特征：year, votes, genre1, genre2
# 标签：rating

# 移除rating列作为标签：将rating重命名为label，原始rating列消失
df_reg = df.withColumnRenamed("rating", "label")

# 重新定义适配回归任务的VectorAssembler：删除不存在的"rating"
assembler_reg = VectorAssembler(
    inputCols=["year", "votes", "genre1_onehot", "genre2_onehot"],
    outputCol="features_raw"
)

# 构建完整的回归Pipeline，替换为新assembler_reg
reg_pipeline = Pipeline(stages=[
    genre1_indexer,
    genre2_indexer,
    genre1_encoder,
    genre2_encoder,
    assembler_reg,
    scaler
])

reg_pipeline_model = reg_pipeline.fit(df_reg)
reg_feature_df = reg_pipeline_model.transform(df_reg)

# 划分数据
reg_train, reg_test = reg_feature_df.randomSplit([0.7, 0.3], seed=42)

# 随机森林回归
rf_reg = RandomForestRegressor(featuresCol="features_scaled", labelCol="label",
                               numTrees=50, maxDepth=8)

rf_reg_model = rf_reg.fit(reg_train)
rf_reg_predictions = rf_reg_model.transform(reg_test)

# 评估回归模型
reg_eval = RegressionEvaluator(labelCol="label", predictionCol="prediction")

rmse = reg_eval.setMetricName("rmse").evaluate(rf_reg_predictions)
mae = reg_eval.setMetricName("mae").evaluate(rf_reg_predictions)
r2 = reg_eval.setMetricName("r2").evaluate(rf_reg_predictions)

print(f"随机森林回归评估结果:")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE: {mae:.4f}")
# R² 通常在 0 到 1 之间，越接近 1 表示模型拟合越好；为负数直接宣告了这个模型毫无预测能力；
print(f"  R²: {r2:.4f}")

# 查看预测结果
rf_reg_predictions.select("title", "label", "prediction").show()

8. 电影评分预测（回归任务）
随机森林回归评估结果:
  RMSE: 0.4303
  MAE: 0.3530
  R²: -0.5112
+--------------------+-----+-----------------+
|               title|label|       prediction|
+--------------------+-----+-----------------+
|        Interstellar|  8.6|            8.707|
|The Shawshank Red...|  9.3|8.701000000000002|
+--------------------+-----+-----------------+

